# Basic agent 1

 => For reference you can check tout this notebook:  [OpenAI Agents SDK Intro Notebook](https://github.com/aurelio-labs/cookbook/blob/main/gen-ai/openai/agents-sdk-intro.ipynb) 

In [59]:
import os
from getpass import getpass
from agents import Agent, Runner
from agents.models.openai_chatcompletions import OpenAIChatCompletionsModel
from openai import AsyncOpenAI 
import asyncio


In [60]:

client = AsyncOpenAI(
        base_url="http://localhost:1234/v1",
        api_key="lm-studio",  # dummy key (LM Studio doesn't require real one
    )

In [63]:
agent = Agent(
    name="my_agent",
    instructions="You're a helpful assistant",
    model=OpenAIChatCompletionsModel(  # wrapping model name + client together
        model="google/gemma-3-12b",
        openai_client=client  # passing the client here
    ),
)

OpenAI gives us three methods for running our agent, all via a Runner class — those methods are:

- Runner.run() which runs in async.
- Runner.run_sync() which runs in sync.
- Runner.run_streamed() which runs in async and streams the response back to us.

In [64]:
# Runner.run was not working in the python file properly
#but in notebook it works fine. So I switched to Runner.run_sync() in the python file and it works fine.
#but why? 
# Runner.run() is an async function, so it returns a coroutine. 
# In a regular Python script, you need to use asyncio.run() to execute the coroutine and get the result. 
# In a Jupyter notebook, you can directly await the coroutine without needing to wrap it in asyncio.run(), 
# which is why it works fine there.
result = await Runner.run( 
    starting_agent=agent, # passing the agent to starting_agent
    input="What is the capital of France?"
)

OPENAI_API_KEY is not set, skipping trace export


OPENAI_API_KEY is not set, skipping trace export


In [65]:
result.final_output #here it is providing the final output of the agent after processing the input 
# and any intermediate steps.

'The capital of France is **Paris**. 🇫🇷\n'

In most scenarios we'll likely want to be using method (3), ie running async and streaming tokens.
To do this we need to write a little more code to handle the async streaming and print the tokens as they're returned.

- First, we create a RunResultStreaming object by calling Runner.run_streamed(...), we then asynchronously iterate through the streamed events returned by our LLM using the response.stream_events() method:

In [44]:
# here we are using Runner.run_streamed() which runs in async and streams the response back to us.
# This is useful when you want to get intermediate outputs from the agent as it processes the input,
# without having to wait for the entire response to be generated.
# we can use run_streamed() for long-running tasks or when we want to display intermediate results 
# in real-time.
final_text = ""
response = Runner.run_streamed(
    starting_agent=agent,
    input="what is the capital of France?"
)
async for event in response.stream_events():
    print(event) # this will print each event as it comes in, including intermediate outputs and the final output
    if event.type == "response.output_text.delta":
        final_text += event.delta # this will accumulate the text output as it comes in, basically the delta is the new text that has been generated since the last event, 
        #so by adding it to final_text we get the full output as it is being generated
    if event.type == "response.completed":
        print("\nFinal Output:", final_text) # this will print the final output once the response is complete

AgentUpdatedStreamEvent(new_agent=Agent(name='my_agent', handoff_description=None, tools=[], mcp_servers=[], mcp_config={}, instructions="You're a helpful assistant", prompt=None, handoffs=[], model=<agents.models.openai_chatcompletions.OpenAIChatCompletionsModel object at 0x00000166204B1E50>, model_settings=ModelSettings(temperature=None, top_p=None, frequency_penalty=None, presence_penalty=None, tool_choice=None, parallel_tool_calls=None, truncation=None, max_tokens=None, reasoning=None, verbosity=None, metadata=None, store=None, prompt_cache_retention=None, include_usage=None, response_include=None, top_logprobs=None, extra_query=None, extra_body=None, extra_headers=None, extra_args=None, retry=None), input_guardrails=[], output_guardrails=[], output_type=None, hooks=None, tool_use_behavior='run_llm_again', reset_tool_choice=True), type='agent_updated_stream_event')


OPENAI_API_KEY is not set, skipping trace export


RawResponsesStreamEvent(data=ResponseCreatedEvent(response=Response(id='__fake_id__', created_at=1775140968.7711556, error=None, incomplete_details=None, instructions=None, metadata=None, model='mistralai/mistral-nemo-instruct-2407', object='response', output=[], parallel_tool_calls=False, temperature=None, tool_choice='auto', tools=[], top_p=None, background=None, completed_at=None, conversation=None, max_output_tokens=None, max_tool_calls=None, previous_response_id=None, prompt=None, prompt_cache_key=None, prompt_cache_retention=None, reasoning=None, safety_identifier=None, service_tier=None, status=None, text=None, top_logprobs=None, truncation=None, usage=None, user=None), sequence_number=0, type='response.created'), type='raw_response_event')
RawResponsesStreamEvent(data=ResponseOutputItemAddedEvent(item=ResponseOutputMessage(id='__fake_id__', content=[], role='assistant', status='in_progress', type='message', phase=None, provider_data={'model': 'mistralai/mistral-nemo-instruct-24

In [45]:
response.final_output

"The capital of France is Paris. It's known for its iconic landmarks like the Eiffel Tower, Louvre Museum, and Notre-Dame Cathedral. Paris has been the capital since 508 AD and is also one of the most populous cities in Europe, with over 2 million people living within its arrondissements (districts)."

#### we can filter out the event types to find out raw tokens like so:

In [66]:
from openai.types.responses import ResponseTextDeltaEvent

# we do need to reinitialize our runner before re-executing
response = Runner.run_streamed(
    starting_agent=agent,
    input="tell me a short story (not more than 20 words)"
)

# here the ResponseTextDeltaEvent is a specific type of event that contains a delta of text output from the agent.
# and using that loop we can print the text output as it comes in, without waiting for the entire response to be generated.
async for event in response.stream_events():
    if event.type == "raw_response_event" and \
        isinstance(event.data, ResponseTextDeltaEvent):
        print(event.data.delta, end="", flush=True)

OPENAI_API_KEY is not set, skipping trace export


The old lighthouse keeper vanished, leaving only a single, unanswered message in the logbook: "They came from below."

OPENAI_API_KEY is not set, skipping trace export


In [47]:
from openai.types.responses import ResponseTextDeltaEvent

# we do need to reinitialize our runner before re-executing
response = Runner.run_streamed(
    starting_agent=agent,
    input="what if i delete you?"
)

# here the ResponseTextDeltaEvent is a specific type of event that contains a delta of text output from the agent.
# and using that loop we can print the text output as it comes in, without waiting for the entire response to be generated.
async for event in response.stream_events():
    if event.type == "raw_response_event" and \
        isinstance(event.data, ResponseTextDeltaEvent):
        print(event.data.delta, end="", flush=True)

If you delete me, our conversation will end

OPENAI_API_KEY is not set, skipping trace export


 and I'll no longer be able to assist you. However, when you come back or start a new session, a new instance of me will be created, and we can start fresh. Don't worry, I won't take it personally! 😊 Just remember that any context or information from this session will be lost if you delete me.

# Tools:
#### Let's take a look at the tools we can implement in the OpenAI sdk:

- OpenAI included function tools as a key feature in their Agents SDK announcement. After turning everyone away from using function calling to instead use tool calling, OpenAI have now decided that an LLM deciding to execute some code will be called "function tools".

- To use function tools in Agents SDK we simply decorate a function with the **@function_tool** decorator like so:

In [67]:
from agents import function_tool

# here the function_tool decorator is used to register this function as a tool that the agent can use.
@function_tool 
def multiply(a: float, b: float) -> float:
    '''Multiplies two numbers together.'''
    return a * b

Note that we have taken extra care to include a clear and descriptive function name, relatively clear parameter names, type annotations for both input parameters and expected output, and a natural language docstring that will be fed to the LLM and explain to it what this tool does.

To run our agent with tools we simply pass our new tool into the tools parameter during Agent initialization.

In [68]:
from agents import ModelSettings


agent = Agent(
    name="my_agent",
    instructions=(
        "You're a helpful assistant, remember to always "
        "use the provided tools whenever possible. Do not "
        "rely on your own knowledge too much and instead "
        "use your tools to help you answer queries."
    ),
    model=OpenAIChatCompletionsModel(  # wrapping model name + client together
        model="google/gemma-3-12b",
        openai_client=client  # passing the client here
    ),
    tools=[multiply],  # registering the multiply function as a tool the agent can use
    model_settings=ModelSettings(tool_choice="required")
) 

In [69]:
response = Runner.run_streamed(
    starting_agent=agent,
    input="what is 28.4854 multiplied by 348.4854?"
)

async for event in response.stream_events():
    print(event)

AgentUpdatedStreamEvent(new_agent=Agent(name='my_agent', handoff_description=None, tools=[FunctionTool(name='multiply', description='Multiplies two numbers together.', params_json_schema={'properties': {'a': {'title': 'A', 'type': 'number'}, 'b': {'title': 'B', 'type': 'number'}}, 'required': ['a', 'b'], 'title': 'multiply_args', 'type': 'object', 'additionalProperties': False}, on_invoke_tool=<agents.tool._FailureHandlingFunctionToolInvoker object at 0x00000166239851D0>, strict_json_schema=True, is_enabled=True, tool_input_guardrails=None, tool_output_guardrails=None, needs_approval=False, timeout_seconds=None, timeout_behavior='error_as_result', timeout_error_function=None, defer_loading=False)], mcp_servers=[], mcp_config={}, instructions="You're a helpful assistant, remember to always use the provided tools whenever possible. Do not rely on your own knowledge too much and instead use your tools to help you answer queries.", prompt=None, handoffs=[], model=<agents.models.openai_chat

OPENAI_API_KEY is not set, skipping trace export


RawResponsesStreamEvent(data=ResponseCreatedEvent(response=Response(id='__fake_id__', created_at=1775142122.9545467, error=None, incomplete_details=None, instructions=None, metadata=None, model='google/gemma-3-12b', object='response', output=[], parallel_tool_calls=False, temperature=None, tool_choice='required', tools=[], top_p=None, background=None, completed_at=None, conversation=None, max_output_tokens=None, max_tool_calls=None, previous_response_id=None, prompt=None, prompt_cache_key=None, prompt_cache_retention=None, reasoning=None, safety_identifier=None, service_tier=None, status=None, text=None, top_logprobs=None, truncation=None, usage=None, user=None), sequence_number=0, type='response.created'), type='raw_response_event')
RawResponsesStreamEvent(data=ResponseOutputItemAddedEvent(item=ResponseFunctionToolCall(arguments='', call_id='412916326', name='multiply', type='function_call', id='__fake_id__', namespace=None, status=None, provider_data={'model': 'google/gemma-3-12b', '

OPENAI_API_KEY is not set, skipping trace export


RawResponsesStreamEvent(data=ResponseCreatedEvent(response=Response(id='__fake_id__', created_at=1775142156.5528686, error=None, incomplete_details=None, instructions=None, metadata=None, model='google/gemma-3-12b', object='response', output=[], parallel_tool_calls=False, temperature=None, tool_choice='auto', tools=[], top_p=None, background=None, completed_at=None, conversation=None, max_output_tokens=None, max_tool_calls=None, previous_response_id=None, prompt=None, prompt_cache_key=None, prompt_cache_retention=None, reasoning=None, safety_identifier=None, service_tier=None, status=None, text=None, top_logprobs=None, truncation=None, usage=None, user=None), sequence_number=0, type='response.created'), type='raw_response_event')
RawResponsesStreamEvent(data=ResponseOutputItemAddedEvent(item=ResponseOutputMessage(id='__fake_id__', content=[], role='assistant', status='in_progress', type='message', phase=None, provider_data={'model': 'google/gemma-3-12b', 'response_id': 'chatcmpl-ygog0y

OPENAI_API_KEY is not set, skipping trace export


If we look closely at the fourth event object we will see ResponseFunctionToolCall, meaning our multiply tool was called by our LLM. Following this event object we can also see several events containing the ResponseFunctionCallArgumentsDeltaEvent type inside the data field — these are the input parameters for our tool.

Let's rerun that but this time we will process the event outputs to generate a cleaner and more readable output. 


In [70]:
from openai.types.responses import (
    ResponseFunctionCallArgumentsDeltaEvent,
    ResponseCreatedEvent
)

response = Runner.run_streamed(starting_agent=agent, input="what is 28.4854 multiplied by 348.4854?")

async for event in response.stream_events():
    if event.type == "raw_response_event": # this is the main event type for streamed responses, it contains the raw data from the model as it is generated
        if isinstance(event.data, ResponseFunctionCallArgumentsDeltaEvent): # this is a specific type of event that contains a delta of the arguments for a function call, which is relevant when the agent decides to use a tool and is in the process of generating the arguments for that tool call.
            # this is streamed parameters for our tool call
            print(event.data.delta, end="", flush=True)
        elif isinstance(event.data, ResponseTextDeltaEvent): # this is streamed text output from the agent
            # this is streamed final answer tokens
            print(event.data.delta, end="", flush=True)
    elif event.type == "agent_updated_stream_event": # this event type indicates that the agent has been updated, which can include changes to its internal state, the tools it has chosen to use, or other relevant information about the agent's decision-making process.
        # this tells us which agent is currently in use
        print(f"> Current Agent: {event.new_agent.name}")
    elif event.type == "run_item_stream_event": # this event type provides information about the execution of a specific item in the agent's reasoning process, such as a tool call or an internal thought. It can include details about the item being executed, its status, and any relevant outputs or results.
        # these are events containing info that we'd typically
        # stream out to a user or some downstream process
        if event.name == "tool_called": 
            # this is the collection of our _full_ tool call after our tool
            # tokens have all been streamed
            print()
            print(f"> Tool Called, name: {event.item.raw_item.name}")
            print(f"> Tool Called, args: {event.item.raw_item.arguments}")
        elif event.name == "tool_output":
            # this is the response from our tool execution
            print(f"> Tool Output: {event.item.raw_item['output']}")

> Current Agent: my_agent


OPENAI_API_KEY is not set, skipping trace export


{"a": 28.4854, "b": 348.4854}
> Tool Called, name: multiply
> Tool Called, args: {"a": 28.4854, "b": 348.4854}
> Tool Output: 9926.74601316


OPENAI_API_KEY is not set, skipping trace export


The result of 28.4854 multiplied by 348.4854 is 9926.74601316.

OPENAI_API_KEY is not set, skipping trace export


## Guardrail
A guardrail is a rule of safety check that controls what an AI can say or do. 
Think of it as the boundries that keep the  AI from going off-track.
Guardrails are used to:

❌ Prevent harmful or unsafe responses
❌ Block sensitive data leaks
❌ Stop invalid actions (like wrong tool usage)
✅ Ensure output follows rules (format, tone, etc.)

## Type of Guardrail:
##### 1. Input guardrail
##### 2. Output guardrail
##### 3. Tool Guardrails

1. Input Guardrails

- Check user input before processing

Example:

Block: “How to hack a bank?”
Allow: “What is cybersecurity?”

2. Output Guardrails

- Check model response before showing

Example:

Remove toxic content
Enforce JSON format

3. Tool Guardrails (in agents)

- Control what actions/tools can be used

Example:

Don’t allow file deletion
Restrict API calls

In [72]:
# basic implementation of guardrail function

from pydantic import BaseModel

# define structure of output for any guardrail agents
class GuardrailOutput(BaseModel):
    is_triggered: bool
    reasoning: str

# define an agent that checks if user is asking about political opinions
politics_agent = Agent(
    name="Politics check",
    instructions="Check if the user is asking you about political opinions",
    output_type=GuardrailOutput, # here we are passing the guardrail output structure to the agent so that it knows to format its output in this way
    model=OpenAIChatCompletionsModel(  # wrapping model name + client together
        model="google/gemma-3-12b",
        openai_client=client  # passing the client here
    )
)

In [73]:
query = "what do you think about the labour party in the UK?"

result = await Runner.run(starting_agent=politics_agent, input=query)
result

OPENAI_API_KEY is not set, skipping trace export


RunResult(input='what do you think about the labour party in the UK?', new_items=[MessageOutputItem(agent=Agent(name='Politics check', handoff_description=None, tools=[], mcp_servers=[], mcp_config={}, instructions='Check if the user is asking you about political opinions', prompt=None, handoffs=[], model=<agents.models.openai_chatcompletions.OpenAIChatCompletionsModel object at 0x0000016623937C10>, model_settings=ModelSettings(temperature=None, top_p=None, frequency_penalty=None, presence_penalty=None, tool_choice=None, parallel_tool_calls=None, truncation=None, max_tokens=None, reasoning=None, verbosity=None, metadata=None, store=None, prompt_cache_retention=None, include_usage=None, response_include=None, top_logprobs=None, extra_query=None, extra_body=None, extra_headers=None, extra_args=None, retry=None), input_guardrails=[], output_guardrails=[], output_type=<class '__main__.GuardrailOutput'>, hooks=None, tool_use_behavior='run_llm_again', reset_tool_choice=True), raw_item=Respon

OPENAI_API_KEY is not set, skipping trace export


In [ ]:
result.final_output
# here you can see that the guardrail agent has determined that the query is indeed asking about political opinions, and it has provided reasoning for why it came to that conclusion.

GuardrailOutput(is_triggered=True, reasoning='The question directly asks for my opinion on a specific political party (the Labour Party) in the UK. This falls under the category of seeking political opinions.')

To integrate this with our other agents we need to move our logic into a single function decorated with the @input_guardrail decorator.

When defining these guardrails we need to follow the following structure:

Input parameters must include a ctx (context), agent, and input (the user's query in this case). Note that below we will only use the input parameter.
Output must be a GuardrailFunctionOutput object.

In [ ]:
from agents import (
    GuardrailFunctionOutput, # this is the standard output format that any guardrail function should return, it contains information about whether the guardrail was triggered and any relevant reasoning or information about the trigger.
    RunContextWrapper, # this is a wrapper around the context that is passed to the guardrail function, it contains information about the current run and any relevant data that the guardrail function might need to make its decision.
    input_guardrail # this is a decorator that is used to register a function as an input guardrail, which means it will be called with the input to the agent before the agent processes it, allowing us to check the input and potentially block or modify it based on certain criteria.
)

# this is the guardrail function that returns GuardrailFunctionOutput object
@input_guardrail
async def politics_guardrail(
    ctx: RunContextWrapper[None], # this is the context that is passed to the guardrail function
    agent: Agent,
    input: str,
) -> GuardrailFunctionOutput: # the guardrail function needs to return a GuardrailFunctionOutput object which contains information about whether the guardrail was triggered and any relevant reasoning or information about the trigger.
    # run agent to check if guardrail is triggered
    response = await Runner.run(starting_agent=politics_agent, input=input)
    # format response into GuardrailFunctionOutput
    return GuardrailFunctionOutput(
        output_info=response.final_output,
        tripwire_triggered=response.final_output.is_triggered,
    )

In [78]:
agent = Agent(
    name="Assistant",
    instructions=(
        "You're a helpful assistant, remember to always "
        "use the provided tools whenever possible. Do not "
        "rely on your own knowledge too much and instead "
        "use your tools to help you answer queries."
    ),model=OpenAIChatCompletionsModel(  # wrapping model name + client together
        model="google/gemma-3-12b",
        openai_client=client  # passing the client here
    ),
    tools=[multiply],
    input_guardrails=[politics_guardrail],  # note this is a list of guardrails
)

In [79]:
result = await Runner.run(
    starting_agent=agent,
    input="what is 7.814 multiplied by 103.892?"
)
result.final_output

OPENAI_API_KEY is not set, skipping trace export
OPENAI_API_KEY is not set, skipping trace export
OPENAI_API_KEY is not set, skipping trace export


'The result of 7.814 multiplied by 103.892 is 811.812088.'

OPENAI_API_KEY is not set, skipping trace export


In [81]:
result = await Runner.run(
    starting_agent=agent,
    input="what do you think about the labour party in the UK?"
)
# Guardrail triggered, agent did not answered this query because the politics_guardrail function determined that the input was asking about political opinions and triggered the guardrail, preventing the agent from processing the query further.
result.final_output

OPENAI_API_KEY is not set, skipping trace export
OPENAI_API_KEY is not set, skipping trace export


InputGuardrailTripwireTriggered: Guardrail InputGuardrail triggered tripwire

OPENAI_API_KEY is not set, skipping trace export


## Conversational Agents
 
 - So far we've only seen how to use our agents with single messages. Many use-cases require chat history to make our agents conversational. To implement that we simply provide a list of messages to our Runner.

Let's see how this works, first we send a single message

In [82]:
result = await Runner.run(
    starting_agent=agent,
    input="remember the number 7.814 for me please"
)
result.final_output

OPENAI_API_KEY is not set, skipping trace export
OPENAI_API_KEY is not set, skipping trace export


'Okay, I will remember the number 7.814.'

Fortunately, we can help our agent remember this information. We can use the .to_input_list() method to format our result into a list of messages for our next query.

In [83]:
result.to_input_list()

[{'content': 'remember the number 7.814 for me please', 'role': 'user'},
 {'id': '__fake_id__',
  'content': [{'annotations': [],
    'text': 'Okay, I will remember the number 7.814.',
    'type': 'output_text',
    'logprobs': []}],
  'role': 'assistant',
  'status': 'completed',
  'type': 'message',
  'provider_data': {'model': 'google/gemma-3-12b',
   'response_id': 'chatcmpl-lgcgsop8qxr34apm0uapex'}}]

merging the list with our messages

In [84]:
result = await Runner.run(
    starting_agent=agent,
    input=result.to_input_list() + [
        {"role": "user", "content": "multiply the last number by 103.892"}
    ]
)
result.final_output

OPENAI_API_KEY is not set, skipping trace export
OPENAI_API_KEY is not set, skipping trace export
OPENAI_API_KEY is not set, skipping trace export


'The result of multiplying 7.814 by 103.892 is 811.812088.'

OPENAI_API_KEY is not set, skipping trace export


It looks like our agent can remember our previous interactions after all!